# Chapter 2 — Golden sets: building, sampling, maintaining

Stratified allocation and its standard errors, weighting an over-sampled risk slice back to traffic, the
size table n = z²p(1−p)/m², and the selection inflation from picking the best of K equally good variants,
predicted by extreme-value theory and measured by simulation. Every cell **asserts** what it checks.

In [ ]:
import math, os, sys, random
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import numpy as np
from evalkit import basics as B

## 2.1 — three allocations of 400 items across four intents

In [ ]:
SIZES = {"billing": 6000, "shipping": 3000, "returns": 800, "account": 200}
SDS = {"billing": 0.3, "shipping": 0.3, "returns": 0.45, "account": 0.5}
for rule in ("proportional", "neyman", "equal"):
    a = B.stratified_allocation(SIZES, SDS, 400, rule)
    print(f"{rule:12s} {a}  overall SE {B.stratified_se(SIZES, SDS, a) * 100:.2f} points")
assert list(B.stratified_allocation(SIZES, SDS, 400, "neyman").values()) == [228, 114, 45, 13]
# Monte Carlo check of the stratified SE formula under proportional allocation
rng = np.random.default_rng(0); N = sum(SIZES.values())
p_true = {"billing": 0.9, "shipping": 0.9, "returns": 0.7, "account": 0.5}
a = B.stratified_allocation(SIZES, SDS, 400, "proportional")
est = [sum(SIZES[h] / N * rng.binomial(a[h], p_true[h]) / a[h] for h in SIZES) for _ in range(20_000)]
print("simulated SE:", round(np.std(est) * 100, 2), " formula:", round(B.stratified_se(SIZES, SDS, a) * 100, 2))
assert abs(np.std(est) * 100 - B.stratified_se(SIZES, SDS, a) * 100) < 0.05

## 2.2 — weighting an over-sampled risk slice back to traffic

In [ ]:
p, se = B.weighted_rate({"normal": (0.95, 300, 276), "risky": (0.05, 300, 240)})
print(f"unweighted {(276 + 240) / 600:.1%}   weighted {p:.1%} ± {1.96 * se * 100:.1f}")
assert round(p, 3) == 0.914 and round(se * 100, 2) == 1.49

## 2.3 — the size table

In [ ]:
for m in (.01, .02, .03, .05, .07):
    print(f"±{m * 100:.0f} points:", [B.n_for_margin(m, p) for p in (.5, .8, .9)])
assert [B.n_for_margin(m) for m in (.01, .02, .03, .05, .07)] == [9604, 2401, 1068, 385, 196]

## 2.5 — selection inflation: predicted by E[max of K normals], then measured

In [ ]:
for K in (2, 5, 10, 20, 50):
    print(K, round(B.expected_max_normal(K), 3), f"+{B.selection_bias(0.7, 200, K) * 100:.1f} points")
# the numerical integral against a direct simulation of the maximum of K normals
sim = np.random.default_rng(1).standard_normal((200_000, 20)).max(axis=1).mean()
assert abs(sim - B.expected_max_normal(20)) < 0.01
meas = B.simulate_selection(0.7, 200, 20)
print("measured inflation, K=20:", round(meas * 100, 2), "points")
assert round(meas * 100, 1) == 5.9 and round(B.selection_bias(0.7, 200, 20) * 100, 1) == 6.1

## Exercises

In [ ]:
sizes = {"en": 85, "es": 10, "de": 5}
sds = {k: math.sqrt(p * (1 - p)) for k, p in {"en": .92, "es": .85, "de": .75}.items()}
for rule in ("proportional", "neyman", "equal"):
    a = B.stratified_allocation(sizes, sds, 600, rule)
    print(rule, a, round(B.stratified_se(sizes, sds, a) * 100, 2))
assert list(B.stratified_allocation(sizes, sds, 600, "neyman").values()) == [481, 74, 45]
p, se = B.weighted_rate({"g": (0.915, 500, 440), "l": (0.08, 200, 140), "i": (0.005, 100, 60)})
assert round(p * 100, 1) == 86.4 and round(se * 100, 2) == 1.35
assert round(B.simulate_selection(0.8, 400, 10) * 100, 1) == 3.0